# CDISC and ADaM Made Practical  
## Simulated Clinical Trial Dataset, SDTM Mapping, ADaM Derivations, TLF/TLG Outputs, and Visual Analysis

**Author:** Mark Ihrwell R. Petalcorin, PhD  

This notebook creates a complete educational clinical-trial programming workflow using simulated data benchmarked to published PubMed-indexed clinical trial literature. It is designed for learning CDISC-style thinking, SDTM mapping, ADaM derivation, traceability, tables, listings, figures, and clinical programming validation.

> **Important note:** The datasets generated here are synthetic. They are not patient-level data from SPRINT, ACCORD, STEP, or any other trial. The simulation parameters are loosely benchmarked to published summary-level values, then adapted for a small teaching dataset. This notebook is for education, portfolio demonstration, and interview preparation, not for clinical decision-making.

## 1. Scientific and Regulatory Benchmarking

The worked example is a two-arm hypertension cardiovascular-risk trial inspired by the structure and published summary values of major blood-pressure outcome trials.

### PubMed-indexed clinical trial references used for simulation assumptions

1. **SPRINT Research Group.** A Randomized Trial of Intensive versus Standard Blood-Pressure Control. *New England Journal of Medicine*. 2015;373:2103-2116. DOI: https://doi.org/10.1056/NEJMoa1511939. PubMed PMID: 26551272.  
   Key benchmark values used here: older high-risk hypertensive population, intensive systolic target below 120 mm Hg, standard target below 140 mm Hg, achieved 1-year SBP approximately 121.4 versus 136.2 mm Hg, lower cardiovascular event risk in intensive therapy, but increased selected safety events.

2. **ACCORD Study Group.** Effects of Intensive Blood-Pressure Control in Type 2 Diabetes Mellitus. *New England Journal of Medicine*. 2010;362:1575-1585. DOI: https://doi.org/10.1056/NEJMoa1001286. PubMed PMID: 20228401.  
   Key benchmark values used here: intensive systolic target below 120 mm Hg versus standard target below 140 mm Hg, achieved 1-year SBP approximately 119.3 versus 133.5 mm Hg in a diabetic population.

3. **Williamson et al.** Intensive vs Standard Blood Pressure Control and Cardiovascular Disease Outcomes in Adults Aged 75 Years or Older. *JAMA*. 2016;315:2673-2682. DOI: https://doi.org/10.1001/jama.2016.7050. PubMed PMID: 27195814.  
   Key benchmark concept used here: older subgroup analysis, with attention to safety and benefit in older adults.

### CDISC and regulatory references used for data-standard concepts

1. **CDISC SDTM.** Study Data Tabulation Model. Official CDISC foundational standard.  
2. **CDISC ADaM.** Analysis Data Model. Official CDISC foundational standard supporting statistical analysis and traceability.  
3. **CDISC Define-XML.** Metadata standard for describing tabular datasets, including SDTM, SEND, and ADaM.  
4. **CDISC Controlled Terminology.** Standard valid values for CDISC-defined datasets.  
5. **FDA Study Data Technical Conformance Guide.** FDA technical specifications for standardized electronic study data submissions.

This notebook uses CDISC-like structures for teaching, but it does not claim full regulatory compliance. True submission packages require formal SDTMIG, ADaMIG, controlled terminology, define.xml, reviewer guides, validation reports, and sponsor-specific standards.

In [ ]:
# ================================================================
# 0. Environment setup
# ================================================================

import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, HTML

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

SEED = 20260624
rng = np.random.default_rng(SEED)

OUTDIR = Path("cdisc_adam_simulated_trial_outputs")
FIGDIR = OUTDIR / "figures"
DATADIR = OUTDIR / "datasets"
OUTDIR.mkdir(exist_ok=True)
FIGDIR.mkdir(exist_ok=True)
DATADIR.mkdir(exist_ok=True)

def savefig(name):
    path = FIGDIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    return path

def pct(n, d):
    return 100 * n / d if d else np.nan

def styled_table(df, caption=None, precision=2):
    sty = df.style.format(precision=precision)
    if caption:
        sty = sty.set_caption(caption)
    return sty

display(Markdown("Notebook environment ready."))

## 2. Simulation Design

We simulate a randomized, open-label, parallel-group hypertension trial with two treatment strategies:

- **Intensive BP Control**, target systolic blood pressure below 120 mm Hg.
- **Standard BP Control**, target systolic blood pressure below 140 mm Hg.

The synthetic study includes:

- Demographics and baseline characteristics.
- Treatment exposure.
- Longitudinal vital signs.
- Selected laboratory safety markers.
- Adverse events.
- Disposition events.
- Cardiovascular endpoint events.

The simulation intentionally includes realistic clinical noise, missing visits, treatment discontinuation, and adverse events, because clinical trial data are never perfectly clean.

In [ ]:
# ================================================================
# 1. Simulation parameters benchmarked to published summary values
# ================================================================

N = 360
STUDYID = "HTN-CDISC-001"
SITE_IDS = [f"{i:03d}" for i in range(101, 113)]
ARMS = ["INTENSIVE BP CONTROL", "STANDARD BP CONTROL"]

# Visit schedule in days
VISITS = [
    ("BASELINE", 0),
    ("WEEK 4", 28),
    ("MONTH 3", 84),
    ("MONTH 6", 168),
    ("MONTH 12", 365),
]

benchmark = pd.DataFrame({
    "Parameter": [
        "SPRINT intensive achieved SBP at 1 year",
        "SPRINT standard achieved SBP at 1 year",
        "ACCORD intensive achieved SBP at 1 year",
        "ACCORD standard achieved SBP at 1 year",
        "Synthetic baseline SBP mean",
        "Synthetic age mean",
        "Synthetic endpoint risk, standard arm",
        "Synthetic endpoint relative reduction, intensive arm"
    ],
    "Value": [
        "121.4 mm Hg",
        "136.2 mm Hg",
        "119.3 mm Hg",
        "133.5 mm Hg",
        "139-141 mm Hg",
        "67-69 years",
        "approximately 7-10% over 12 months",
        "approximately 20-30% lower than standard"
    ],
    "Source_or_Rationale": [
        "SPRINT Research Group, NEJM 2015",
        "SPRINT Research Group, NEJM 2015",
        "ACCORD Study Group, NEJM 2010",
        "ACCORD Study Group, NEJM 2010",
        "Teaching simulation calibrated to hypertensive high-risk population",
        "Teaching simulation calibrated to older high-risk population",
        "Teaching simulation scaled from cardiovascular outcome trial event rates",
        "Teaching simulation approximating SPRINT-like treatment benefit"
    ]
})
display(styled_table(benchmark, "Benchmark assumptions used for simulation", precision=1))

## 3. Raw Clinical Trial Data

This section creates source-like raw datasets. In a real trial, these would come from eCRFs, laboratory vendors, randomization systems, safety systems, and clinical adjudication systems.

The raw datasets are deliberately not fully CDISC-standardized. They use readable clinical fields such as `patient_id`, `visit_name`, `systolic_bp`, and `event_description`. Later sections map these into SDTM-like and ADaM-like datasets.

In [ ]:
# ================================================================
# 2. Simulate raw demographics
# ================================================================

subject_nums = np.arange(1, N + 1)
sites = rng.choice(SITE_IDS, size=N)
raw_subject_id = [f"{site}-{num:04d}" for site, num in zip(sites, subject_nums)]
usubjid = [f"{STUDYID}-{sid}" for sid in raw_subject_id]

trt = np.array(ARMS * (N // 2))
if len(trt) < N:
    trt = np.append(trt, rng.choice(ARMS, N - len(trt)))
rng.shuffle(trt)

age = np.clip(rng.normal(68, 8, N).round(), 50, 90).astype(int)
sex = rng.choice(["Male", "Female"], size=N, p=[0.58, 0.42])
race = rng.choice(["White", "Black or African American", "Asian", "Other"], size=N, p=[0.58, 0.28, 0.08, 0.06])
country = rng.choice(["United Kingdom", "United States", "Canada"], size=N, p=[0.40, 0.45, 0.15])
diabetes = rng.choice(["Yes", "No"], size=N, p=[0.25, 0.75])
ckd = rng.choice(["Yes", "No"], size=N, p=[0.18, 0.82])
prior_cvd = rng.choice(["Yes", "No"], size=N, p=[0.22, 0.78])

rand_base = pd.Timestamp("2026-01-10")
rand_offsets = rng.integers(0, 45, N)
randdt = [rand_base + pd.Timedelta(days=int(x)) for x in rand_offsets]
first_dose = [d + pd.Timedelta(days=int(rng.integers(0, 2))) for d in randdt]

raw_dm = pd.DataFrame({
    "study_id": STUDYID,
    "patient_id": raw_subject_id,
    "unique_subject_id": usubjid,
    "site_id": sites,
    "randomized_group": trt,
    "age_years": age,
    "sex": sex,
    "race": race,
    "country": country,
    "diabetes_history": diabetes,
    "chronic_kidney_disease": ckd,
    "prior_cardiovascular_disease": prior_cvd,
    "randomization_date": randdt,
    "first_dose_date": first_dose
})

display(styled_table(raw_dm.head(10), "Raw demographics, first 10 records", precision=1))
raw_dm.to_csv(DATADIR / "raw_demographics.csv", index=False)

In [ ]:
# ================================================================
# 3. Simulate raw exposure
# ================================================================

raw_ex_rows = []
for _, row in raw_dm.iterrows():
    subj = row["unique_subject_id"]
    arm = row["randomized_group"]
    start = row["first_dose_date"]
    discontinued = rng.random() < (0.10 if arm == "INTENSIVE BP CONTROL" else 0.07)
    disc_day = int(rng.integers(60, 360)) if discontinued else 365
    end = start + pd.Timedelta(days=disc_day)
    n_meds_base = rng.integers(1, 4)
    n_meds_follow = n_meds_base + (rng.integers(1, 3) if arm == "INTENSIVE BP CONTROL" else rng.integers(0, 2))
    raw_ex_rows.append({
        "patient_id": row["patient_id"],
        "unique_subject_id": subj,
        "treatment_strategy": arm,
        "exposure_start_date": start,
        "exposure_end_date": end,
        "days_exposed": disc_day + 1,
        "baseline_antihypertensive_count": int(n_meds_base),
        "followup_antihypertensive_count": int(n_meds_follow),
        "treatment_discontinued": "Yes" if discontinued else "No"
    })

raw_ex = pd.DataFrame(raw_ex_rows)
display(styled_table(raw_ex.head(10), "Raw exposure, first 10 records", precision=1))
raw_ex.to_csv(DATADIR / "raw_exposure.csv", index=False)

In [ ]:
# ================================================================
# 4. Simulate raw vital signs
# ================================================================

raw_vs_rows = []
for _, row in raw_dm.iterrows():
    subj = row["unique_subject_id"]
    arm = row["randomized_group"]
    base_sbp = np.clip(rng.normal(140, 10), 125, 175)
    base_dbp = np.clip(rng.normal(78, 8), 55, 105)
    bmi = np.clip(rng.normal(29, 5), 18, 45)

    for visit, day in VISITS:
        # Missing follow-up is possible, especially after baseline
        if visit != "BASELINE" and rng.random() < 0.035:
            continue

        if visit == "BASELINE":
            target_sbp = base_sbp
        else:
            # SPRINT-like achieved 1-year values, with noisy path toward target
            if arm == "INTENSIVE BP CONTROL":
                final_mean = 121.4
            else:
                final_mean = 136.2
            progress = min(day / 365, 1.0)
            target_sbp = base_sbp * (1 - progress) + final_mean * progress

        sbp = np.clip(rng.normal(target_sbp, 7), 90, 190)
        dbp = np.clip(rng.normal(base_dbp - (base_sbp - sbp) * 0.25, 6), 45, 110)
        pulse = np.clip(rng.normal(72, 10), 45, 120)
        dt = row["first_dose_date"] + pd.Timedelta(days=day)

        raw_vs_rows.extend([
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "measurement_date": dt, "test_name": "Systolic Blood Pressure", "value": round(sbp, 1), "unit": "mmHg"},
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "measurement_date": dt, "test_name": "Diastolic Blood Pressure", "value": round(dbp, 1), "unit": "mmHg"},
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "measurement_date": dt, "test_name": "Pulse Rate", "value": round(pulse, 1), "unit": "beats/min"},
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "measurement_date": dt, "test_name": "Body Mass Index", "value": round(bmi + rng.normal(0, 0.3), 1), "unit": "kg/m2"},
        ])

raw_vs = pd.DataFrame(raw_vs_rows)
display(styled_table(raw_vs.head(12), "Raw vital signs, first 12 records", precision=1))
raw_vs.to_csv(DATADIR / "raw_vital_signs.csv", index=False)

In [ ]:
# ================================================================
# 5. Simulate raw laboratory safety data
# ================================================================

raw_lb_rows = []
for _, row in raw_dm.iterrows():
    subj = row["unique_subject_id"]
    arm = row["randomized_group"]
    has_ckd = row["chronic_kidney_disease"] == "Yes"
    base_creat = np.clip(rng.normal(1.05 + (0.25 if has_ckd else 0), 0.20), 0.55, 2.20)
    base_k = np.clip(rng.normal(4.2, 0.35), 3.2, 5.7)
    base_sodium = np.clip(rng.normal(140, 2.5), 132, 148)

    for visit, day in VISITS:
        if visit != "BASELINE" and rng.random() < 0.04:
            continue

        intensive_shift = 0.08 if arm == "INTENSIVE BP CONTROL" and day > 0 else 0
        creat = np.clip(rng.normal(base_creat + intensive_shift * (day / 365), 0.12), 0.50, 3.00)
        potassium = np.clip(rng.normal(base_k + (0.05 if arm == "INTENSIVE BP CONTROL" else 0), 0.28), 2.8, 6.3)
        sodium = np.clip(rng.normal(base_sodium - (0.4 if arm == "INTENSIVE BP CONTROL" else 0), 2.0), 128, 150)
        dt = row["first_dose_date"] + pd.Timedelta(days=day)

        raw_lb_rows.extend([
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "collection_date": dt, "lab_test": "Creatinine", "result": round(creat, 2), "unit": "mg/dL", "lower_limit": 0.60, "upper_limit": 1.30},
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "collection_date": dt, "lab_test": "Potassium", "result": round(potassium, 2), "unit": "mmol/L", "lower_limit": 3.50, "upper_limit": 5.10},
            {"patient_id": row["patient_id"], "unique_subject_id": subj, "visit_name": visit, "visit_day": day, "collection_date": dt, "lab_test": "Sodium", "result": round(sodium, 1), "unit": "mmol/L", "lower_limit": 135.0, "upper_limit": 145.0},
        ])

raw_lb = pd.DataFrame(raw_lb_rows)
display(styled_table(raw_lb.head(12), "Raw laboratory safety data, first 12 records", precision=2))
raw_lb.to_csv(DATADIR / "raw_laboratory.csv", index=False)

In [ ]:
# ================================================================
# 6. Simulate raw adverse events and endpoint events
# ================================================================

ae_terms = [
    ("Dizziness", "NERVOUS SYSTEM DISORDERS", 0.12, 0.16),
    ("Syncope", "NERVOUS SYSTEM DISORDERS", 0.025, 0.045),
    ("Hypotension", "VASCULAR DISORDERS", 0.035, 0.075),
    ("Acute Kidney Injury", "RENAL AND URINARY DISORDERS", 0.025, 0.065),
    ("Electrolyte Imbalance", "METABOLISM AND NUTRITION DISORDERS", 0.030, 0.060),
    ("Fall", "INJURY, POISONING AND PROCEDURAL COMPLICATIONS", 0.050, 0.060),
    ("Headache", "NERVOUS SYSTEM DISORDERS", 0.120, 0.120),
    ("Fatigue", "GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS", 0.100, 0.110),
    ("Cough", "RESPIRATORY, THORACIC AND MEDIASTINAL DISORDERS", 0.080, 0.090),
]

raw_ae_rows = []
raw_ce_rows = []
for _, row in raw_dm.iterrows():
    subj = row["unique_subject_id"]
    arm = row["randomized_group"]
    first = row["first_dose_date"]
    ae_seq = 1

    for term, soc, p_std, p_int in ae_terms:
        p = p_int if arm == "INTENSIVE BP CONTROL" else p_std
        if rng.random() < p:
            start_day = int(rng.integers(1, 360))
            duration = int(rng.integers(1, 28))
            severity = rng.choice(["MILD", "MODERATE", "SEVERE"], p=[0.55, 0.35, 0.10])
            serious = "Y" if (term in ["Syncope", "Acute Kidney Injury"] and severity == "SEVERE" and rng.random() < 0.45) else "N"
            related = rng.choice(["NOT RELATED", "POSSIBLY RELATED", "RELATED"], p=[0.45, 0.40, 0.15])
            outcome = rng.choice(["RECOVERED/RESOLVED", "RECOVERING/RESOLVING", "NOT RECOVERED/NOT RESOLVED"], p=[0.72, 0.18, 0.10])
            raw_ae_rows.append({
                "patient_id": row["patient_id"],
                "unique_subject_id": subj,
                "event_sequence": ae_seq,
                "event_description": term,
                "body_system": soc,
                "start_date": first + pd.Timedelta(days=start_day),
                "end_date": first + pd.Timedelta(days=start_day + duration),
                "severity": severity,
                "serious_event": serious,
                "relationship_to_treatment": related,
                "outcome": outcome
            })
            ae_seq += 1

    # Endpoint events, lower probability in intensive arm
    endpoint_risk = 0.070 if arm == "INTENSIVE BP CONTROL" else 0.095
    if rng.random() < endpoint_risk:
        event = rng.choice(["Myocardial Infarction", "Stroke", "Heart Failure", "Cardiovascular Death"], p=[0.35, 0.25, 0.25, 0.15])
        day = int(rng.integers(30, 365))
        raw_ce_rows.append({
            "patient_id": row["patient_id"],
            "unique_subject_id": subj,
            "event_term": event,
            "event_category": "MAJOR ADVERSE CARDIOVASCULAR EVENT",
            "event_date": first + pd.Timedelta(days=day),
            "fatal_event": "Y" if event == "Cardiovascular Death" else "N"
        })

raw_ae = pd.DataFrame(raw_ae_rows)
raw_ce = pd.DataFrame(raw_ce_rows)

display(styled_table(raw_ae.head(15), "Raw adverse event listing, first 15 records", precision=1))
display(styled_table(raw_ce.head(10), "Raw cardiovascular endpoint listing, first 10 records", precision=1))

raw_ae.to_csv(DATADIR / "raw_adverse_events.csv", index=False)
raw_ce.to_csv(DATADIR / "raw_clinical_events.csv", index=False)

In [ ]:
# ================================================================
# 7. Simulate disposition
# ================================================================

raw_ds_rows = []
ae_discontinue_subjects = set(raw_ae.loc[(raw_ae["severity"] == "SEVERE") & (raw_ae["relationship_to_treatment"] != "NOT RELATED"), "unique_subject_id"].sample(frac=0.25, random_state=SEED).tolist()) if len(raw_ae) else set()
death_subjects = set(raw_ce.loc[raw_ce.get("fatal_event", pd.Series(dtype=str)) == "Y", "unique_subject_id"].tolist()) if len(raw_ce) else set()

for _, row in raw_dm.iterrows():
    subj = row["unique_subject_id"]
    first = row["first_dose_date"]
    if subj in death_subjects:
        status = "DEATH"
        day = int((raw_ce.loc[raw_ce["unique_subject_id"] == subj, "event_date"].iloc[0] - first).days)
    elif subj in ae_discontinue_subjects:
        status = "WITHDRAWAL DUE TO ADVERSE EVENT"
        day = int(rng.integers(60, 340))
    elif rng.random() < 0.045:
        status = "WITHDRAWAL BY SUBJECT"
        day = int(rng.integers(30, 360))
    elif rng.random() < 0.020:
        status = "LOST TO FOLLOW-UP"
        day = int(rng.integers(90, 365))
    else:
        status = "COMPLETED"
        day = 365

    raw_ds_rows.append({
        "patient_id": row["patient_id"],
        "unique_subject_id": subj,
        "disposition_event": status,
        "disposition_date": first + pd.Timedelta(days=day),
        "study_day": day
    })

raw_ds = pd.DataFrame(raw_ds_rows)
display(styled_table(raw_ds.head(15), "Raw disposition, first 15 records", precision=1))
raw_ds.to_csv(DATADIR / "raw_disposition.csv", index=False)

## 4. SDTM-Like Datasets

The next step maps the raw clinical datasets into CDISC SDTM-like domains. These are simplified but intentionally use familiar SDTM-style variable names.

Created domains:

- `DM`, demographics.
- `EX`, exposure.
- `VS`, vital signs.
- `LB`, laboratory tests.
- `AE`, adverse events.
- `DS`, disposition.
- `CE`, clinical events.

This section illustrates the core CDISC idea: raw data are reorganised into predictable domain structures.

In [ ]:
# ================================================================
# 8. Create SDTM-like domains
# ================================================================

# DM
dm = pd.DataFrame({
    "STUDYID": raw_dm["study_id"],
    "DOMAIN": "DM",
    "USUBJID": raw_dm["unique_subject_id"],
    "SUBJID": raw_dm["patient_id"],
    "SITEID": raw_dm["site_id"],
    "AGE": raw_dm["age_years"],
    "AGEU": "YEARS",
    "SEX": raw_dm["sex"].str.upper().str[0],
    "RACE": raw_dm["race"].str.upper(),
    "COUNTRY": raw_dm["country"].map({"United Kingdom": "GBR", "United States": "USA", "Canada": "CAN"}),
    "ARM": raw_dm["randomized_group"],
    "ACTARM": raw_dm["randomized_group"],
    "RFSTDTC": pd.to_datetime(raw_dm["first_dose_date"]).dt.strftime("%Y-%m-%d"),
    "RFXSTDTC": pd.to_datetime(raw_dm["first_dose_date"]).dt.strftime("%Y-%m-%d"),
    "DMDTC": pd.to_datetime(raw_dm["randomization_date"]).dt.strftime("%Y-%m-%d")
})

# EX
ex = pd.DataFrame({
    "STUDYID": STUDYID,
    "DOMAIN": "EX",
    "USUBJID": raw_ex["unique_subject_id"],
    "EXSEQ": np.arange(1, len(raw_ex) + 1),
    "EXTRT": raw_ex["treatment_strategy"],
    "EXDOSE": raw_ex["followup_antihypertensive_count"],
    "EXDOSU": "NUMBER OF ANTIHYPERTENSIVE MEDICATIONS",
    "EXSTDTC": pd.to_datetime(raw_ex["exposure_start_date"]).dt.strftime("%Y-%m-%d"),
    "EXENDTC": pd.to_datetime(raw_ex["exposure_end_date"]).dt.strftime("%Y-%m-%d"),
    "EXDUR": raw_ex["days_exposed"],
})

# VS
vs_testcd = {
    "Systolic Blood Pressure": "SYSBP",
    "Diastolic Blood Pressure": "DIABP",
    "Pulse Rate": "PULSE",
    "Body Mass Index": "BMI"
}
vs = raw_vs.copy()
vs["STUDYID"] = STUDYID
vs["DOMAIN"] = "VS"
vs["VSSEQ"] = vs.groupby("unique_subject_id").cumcount() + 1
vs["VSTEST"] = vs["test_name"].str.upper()
vs["VSTESTCD"] = vs["test_name"].map(vs_testcd)
vs["VSORRES"] = vs["value"]
vs["VSORRESU"] = vs["unit"]
vs["VSSTRESN"] = vs["value"]
vs["VSSTRESU"] = vs["unit"]
vs["VISIT"] = vs["visit_name"]
vs["VISITNUM"] = vs["visit_day"].map({0:1, 28:2, 84:3, 168:4, 365:5})
vs["VSDTC"] = pd.to_datetime(vs["measurement_date"]).dt.strftime("%Y-%m-%d")
vs = vs.rename(columns={"unique_subject_id": "USUBJID"})[[
    "STUDYID","DOMAIN","USUBJID","VSSEQ","VSTESTCD","VSTEST","VSORRES","VSORRESU","VSSTRESN","VSSTRESU","VISIT","VISITNUM","VSDTC"
]]

# LB
lb_testcd = {"Creatinine": "CREAT", "Potassium": "K", "Sodium": "SODIUM"}
lb = raw_lb.copy()
lb["STUDYID"] = STUDYID
lb["DOMAIN"] = "LB"
lb["LBSEQ"] = lb.groupby("unique_subject_id").cumcount() + 1
lb["LBTEST"] = lb["lab_test"].str.upper()
lb["LBTESTCD"] = lb["lab_test"].map(lb_testcd)
lb["LBORRES"] = lb["result"]
lb["LBORRESU"] = lb["unit"]
lb["LBSTRESN"] = lb["result"]
lb["LBSTRESU"] = lb["unit"]
lb["LBNRIND"] = np.where(lb["result"] < lb["lower_limit"], "LOW", np.where(lb["result"] > lb["upper_limit"], "HIGH", "NORMAL"))
lb["VISIT"] = lb["visit_name"]
lb["VISITNUM"] = lb["visit_day"].map({0:1, 28:2, 84:3, 168:4, 365:5})
lb["LBDTC"] = pd.to_datetime(lb["collection_date"]).dt.strftime("%Y-%m-%d")
lb = lb.rename(columns={"unique_subject_id": "USUBJID"})[[
    "STUDYID","DOMAIN","USUBJID","LBSEQ","LBTESTCD","LBTEST","LBORRES","LBORRESU","LBSTRESN","LBSTRESU","LBNRIND","VISIT","VISITNUM","LBDTC"
]]

# AE
ae = raw_ae.copy()
if len(ae):
    ae["STUDYID"] = STUDYID
    ae["DOMAIN"] = "AE"
    ae["AESEQ"] = ae["event_sequence"]
    ae["AETERM"] = ae["event_description"].str.upper()
    ae["AEBODSYS"] = ae["body_system"]
    ae["AESEV"] = ae["severity"]
    ae["AESER"] = ae["serious_event"]
    ae["AEREL"] = ae["relationship_to_treatment"]
    ae["AEOUT"] = ae["outcome"]
    ae["AESTDTC"] = pd.to_datetime(ae["start_date"]).dt.strftime("%Y-%m-%d")
    ae["AEENDTC"] = pd.to_datetime(ae["end_date"]).dt.strftime("%Y-%m-%d")
    ae = ae.rename(columns={"unique_subject_id": "USUBJID"})[[
        "STUDYID","DOMAIN","USUBJID","AESEQ","AETERM","AEBODSYS","AESEV","AESER","AEREL","AEOUT","AESTDTC","AEENDTC"
    ]]
else:
    ae = pd.DataFrame(columns=["STUDYID","DOMAIN","USUBJID","AESEQ","AETERM","AEBODSYS","AESEV","AESER","AEREL","AEOUT","AESTDTC","AEENDTC"])

# DS
ds = raw_ds.copy()
ds["STUDYID"] = STUDYID
ds["DOMAIN"] = "DS"
ds["DSSEQ"] = 1
ds["DSTERM"] = ds["disposition_event"]
ds["DSDECOD"] = ds["disposition_event"]
ds["DSCAT"] = "DISPOSITION EVENT"
ds["DSSTDTC"] = pd.to_datetime(ds["disposition_date"]).dt.strftime("%Y-%m-%d")
ds = ds.rename(columns={"unique_subject_id": "USUBJID"})[["STUDYID","DOMAIN","USUBJID","DSSEQ","DSTERM","DSDECOD","DSCAT","DSSTDTC"]]

# CE
ce = raw_ce.copy()
if len(ce):
    ce["STUDYID"] = STUDYID
    ce["DOMAIN"] = "CE"
    ce["CESEQ"] = ce.groupby("unique_subject_id").cumcount() + 1
    ce["CETERM"] = ce["event_term"].str.upper()
    ce["CECAT"] = ce["event_category"]
    ce["CESTDTC"] = pd.to_datetime(ce["event_date"]).dt.strftime("%Y-%m-%d")
    ce["CEOUT"] = np.where(ce["fatal_event"] == "Y", "FATAL", "NON-FATAL")
    ce = ce.rename(columns={"unique_subject_id": "USUBJID"})[["STUDYID","DOMAIN","USUBJID","CESEQ","CETERM","CECAT","CESTDTC","CEOUT"]]
else:
    ce = pd.DataFrame(columns=["STUDYID","DOMAIN","USUBJID","CESEQ","CETERM","CECAT","CESTDTC","CEOUT"])

domains = {"dm": dm, "ex": ex, "vs": vs, "lb": lb, "ae": ae, "ds": ds, "ce": ce}
for name, df in domains.items():
    df.to_csv(DATADIR / f"sdtm_{name}.csv", index=False)

display(Markdown("### SDTM-like domain row counts"))
sdtm_counts = pd.DataFrame({"Domain": list(domains.keys()), "Rows": [len(x) for x in domains.values()]})
display(styled_table(sdtm_counts, "SDTM-like domain inventory", precision=0))

display(Markdown("### Example SDTM AE"))
display(styled_table(ae.head(10), "SDTM-like AE, first 10 records", precision=1))

## 5. ADaM-Like Analysis Datasets

This section derives ADaM-like datasets from the SDTM-like domains.

Created analysis datasets:

- `ADSL`, subject-level analysis dataset.
- `ADVS`, vital signs analysis dataset.
- `ADLB`, laboratory analysis dataset.
- `ADAE`, adverse-event analysis dataset.
- `ADTTE`, time-to-event endpoint dataset.

The key ADaM concept is **traceability**. Analysis values should be traceable back to SDTM and raw source data.

In [ ]:
# ================================================================
# 9. Create ADaM-like ADSL
# ================================================================

adsl = dm.copy()
adsl["TRT01A"] = adsl["ACTARM"]
adsl["TRTSDT"] = pd.to_datetime(adsl["RFSTDTC"])
adsl["TRTEDT"] = adsl["USUBJID"].map(pd.to_datetime(ex.set_index("USUBJID")["EXENDTC"]))
adsl["SAFFL"] = "Y"
adsl["ITTFL"] = "Y"
adsl["AGEGR1"] = np.where(adsl["AGE"] >= 75, ">=75 YEARS", "<75 YEARS")

risk_map = raw_dm.set_index("unique_subject_id")[["diabetes_history", "chronic_kidney_disease", "prior_cardiovascular_disease"]]
adsl = adsl.merge(risk_map, left_on="USUBJID", right_index=True, how="left")
adsl["DIABFL"] = np.where(adsl["diabetes_history"] == "Yes", "Y", "N")
adsl["CKDFL"] = np.where(adsl["chronic_kidney_disease"] == "Yes", "Y", "N")
adsl["CVHISTFL"] = np.where(adsl["prior_cardiovascular_disease"] == "Yes", "Y", "N")

final_ds = ds.set_index("USUBJID")[["DSDECOD","DSSTDTC"]]
adsl = adsl.merge(final_ds, left_on="USUBJID", right_index=True, how="left")
adsl["EOSSTT"] = adsl["DSDECOD"]
adsl["EOSDT"] = pd.to_datetime(adsl["DSSTDTC"])

adsl = adsl[[
    "STUDYID","USUBJID","SUBJID","SITEID","AGE","AGEU","AGEGR1","SEX","RACE","COUNTRY",
    "ARM","ACTARM","TRT01A","TRTSDT","TRTEDT","SAFFL","ITTFL","DIABFL","CKDFL","CVHISTFL","EOSSTT","EOSDT"
]]

display(styled_table(adsl.head(10), "ADSL, first 10 subjects", precision=1))
adsl.to_csv(DATADIR / "adam_adsl.csv", index=False)

In [ ]:
# ================================================================
# 10. Create ADVS with baseline, change from baseline, and visit variables
# ================================================================

advs = vs.copy()
advs = advs.merge(adsl[["USUBJID","TRT01A","TRTSDT","SAFFL","ITTFL"]], on="USUBJID", how="left")
advs["ADT"] = pd.to_datetime(advs["VSDTC"])
advs["PARAMCD"] = advs["VSTESTCD"]
advs["PARAM"] = advs["VSTEST"]
advs["AVAL"] = advs["VSSTRESN"]
advs["AVALU"] = advs["VSSTRESU"]
advs["AVISIT"] = advs["VISIT"]
advs["AVISITN"] = advs["VISITNUM"]
advs["ABLFL"] = np.where(advs["AVISIT"] == "BASELINE", "Y", "")

base_vs = advs.loc[advs["ABLFL"] == "Y", ["USUBJID","PARAMCD","AVAL"]].rename(columns={"AVAL":"BASE"})
advs = advs.merge(base_vs, on=["USUBJID","PARAMCD"], how="left")
advs["CHG"] = advs["AVAL"] - advs["BASE"]
advs["PCHG"] = 100 * advs["CHG"] / advs["BASE"]

advs = advs[[
    "STUDYID","USUBJID","TRT01A","SAFFL","ITTFL","PARAMCD","PARAM","AVAL","AVALU","BASE","CHG","PCHG",
    "AVISIT","AVISITN","ADT","ABLFL"
]]

display(styled_table(advs.head(12), "ADVS, first 12 records", precision=2))
advs.to_csv(DATADIR / "adam_advs.csv", index=False)

In [ ]:
# ================================================================
# 11. Create ADLB with abnormal flags and change from baseline
# ================================================================

adlb = lb.copy()
adlb = adlb.merge(adsl[["USUBJID","TRT01A","TRTSDT","SAFFL","ITTFL"]], on="USUBJID", how="left")
adlb["ADT"] = pd.to_datetime(adlb["LBDTC"])
adlb["PARAMCD"] = adlb["LBTESTCD"]
adlb["PARAM"] = adlb["LBTEST"]
adlb["AVAL"] = adlb["LBSTRESN"]
adlb["AVALU"] = adlb["LBSTRESU"]
adlb["ANRIND"] = adlb["LBNRIND"]
adlb["AVISIT"] = adlb["VISIT"]
adlb["AVISITN"] = adlb["VISITNUM"]
adlb["ABLFL"] = np.where(adlb["AVISIT"] == "BASELINE", "Y", "")

base_lb = adlb.loc[adlb["ABLFL"] == "Y", ["USUBJID","PARAMCD","AVAL"]].rename(columns={"AVAL":"BASE"})
adlb = adlb.merge(base_lb, on=["USUBJID","PARAMCD"], how="left")
adlb["CHG"] = adlb["AVAL"] - adlb["BASE"]
adlb["PCHG"] = 100 * adlb["CHG"] / adlb["BASE"]

adlb = adlb[[
    "STUDYID","USUBJID","TRT01A","SAFFL","PARAMCD","PARAM","AVAL","AVALU","BASE","CHG","PCHG","ANRIND","AVISIT","AVISITN","ADT","ABLFL"
]]

display(styled_table(adlb.head(12), "ADLB, first 12 records", precision=2))
adlb.to_csv(DATADIR / "adam_adlb.csv", index=False)

In [ ]:
# ================================================================
# 12. Create ADAE with treatment-emergent and analysis flags
# ================================================================

if len(ae):
    adae = ae.copy()
    adae = adae.merge(adsl[["USUBJID","TRT01A","TRTSDT","TRTEDT","SAFFL","AGEGR1","DIABFL","CKDFL"]], on="USUBJID", how="left")
    adae["ASTDT"] = pd.to_datetime(adae["AESTDTC"])
    adae["AENDT"] = pd.to_datetime(adae["AEENDTC"])
    adae["TRTEMFL"] = np.where((adae["ASTDT"] >= adae["TRTSDT"]) & (adae["ASTDT"] <= adae["TRTEDT"] + pd.Timedelta(days=30)), "Y", "N")
    adae["AOCCFL"] = "Y"
    adae["AREL"] = adae["AEREL"]
    adae["ASEV"] = adae["AESEV"]
    adae["AESOC"] = adae["AEBODSYS"]
    adae["AEDECOD"] = adae["AETERM"]
    adae = adae[[
        "STUDYID","USUBJID","TRT01A","SAFFL","AGEGR1","DIABFL","CKDFL","AESEQ","AEDECOD","AESOC","ASEV","AESER","AREL",
        "ASTDT","AENDT","TRTEMFL","AOCCFL","AEOUT"
    ]]
else:
    adae = pd.DataFrame()

display(styled_table(adae.head(15), "ADAE, first 15 records", precision=1))
adae.to_csv(DATADIR / "adam_adae.csv", index=False)

In [ ]:
# ================================================================
# 13. Create ADTTE for time-to-first cardiovascular event
# ================================================================

ce_first = ce.copy()
if len(ce_first):
    ce_first["ADT"] = pd.to_datetime(ce_first["CESTDTC"])
    ce_first = ce_first.sort_values(["USUBJID","ADT"]).groupby("USUBJID").first().reset_index()[["USUBJID","ADT","CETERM","CEOUT"]]
else:
    ce_first = pd.DataFrame(columns=["USUBJID","ADT","CETERM","CEOUT"])

adtte = adsl[["STUDYID","USUBJID","TRT01A","ITTFL","SAFFL","TRTSDT","EOSDT","EOSSTT"]].copy()
adtte = adtte.merge(ce_first, on="USUBJID", how="left")
adtte["CNSR"] = np.where(adtte["ADT"].notna(), 0, 1)
adtte["EVENT"] = np.where(adtte["CNSR"] == 0, "MAJOR ADVERSE CARDIOVASCULAR EVENT", "CENSORED")
adtte["EVNTDESC"] = adtte["CETERM"].fillna("")
adtte["CNSDT"] = adtte["EOSDT"]
adtte["ADT"] = adtte["ADT"].fillna(adtte["CNSDT"])
adtte["AVAL"] = (adtte["ADT"] - adtte["TRTSDT"]).dt.days + 1
adtte["PARAMCD"] = "MACE"
adtte["PARAM"] = "Time to first major adverse cardiovascular event"

adtte = adtte[["STUDYID","USUBJID","TRT01A","ITTFL","PARAMCD","PARAM","AVAL","CNSR","EVENT","EVNTDESC","ADT"]]
display(styled_table(adtte.head(12), "ADTTE, first 12 records", precision=1))
adtte.to_csv(DATADIR / "adam_adtte.csv", index=False)

## 6. Tables, Listings, and Figures

This section creates typical clinical-programming outputs.

The outputs are not intended to replace a formal clinical study report. They show how CDISC-like SDTM and ADaM datasets can be used to produce standard summaries:

- Table 1, baseline characteristics.
- Table 2, systolic blood pressure over visits.
- Table 3, adverse-event summary.
- Table 4, cardiovascular endpoint summary.
- Listings for adverse events and disposition.
- Figures for subject flow, BP trajectory, change from baseline, safety events, and laboratory abnormalities.

In [ ]:
# ================================================================
# 14. Table 1, baseline characteristics
# ================================================================

def summarize_cont(df, var, by="TRT01A"):
    out = df.groupby(by)[var].agg(["count","mean","std","median","min","max"]).reset_index()
    out["Mean (SD)"] = out["mean"].round(1).astype(str) + " (" + out["std"].round(1).astype(str) + ")"
    out["Median [Min, Max]"] = out["median"].round(1).astype(str) + " [" + out["min"].round(1).astype(str) + ", " + out["max"].round(1).astype(str) + "]"
    return out[[by,"count","Mean (SD)","Median [Min, Max]"]]

def summarize_cat(df, var, by="TRT01A"):
    tmp = df.groupby([by, var]).size().reset_index(name="n")
    den = df.groupby(by).size().reset_index(name="N")
    tmp = tmp.merge(den, on=by)
    tmp["n (%)"] = tmp["n"].astype(str) + " (" + (100*tmp["n"]/tmp["N"]).round(1).astype(str) + "%)"
    return tmp[[by, var, "n (%)"]]

table1_age = summarize_cont(adsl, "AGE")
table1_sex = summarize_cat(adsl, "SEX")
table1_agegrp = summarize_cat(adsl, "AGEGR1")
table1_diab = summarize_cat(adsl, "DIABFL")
table1_ckd = summarize_cat(adsl, "CKDFL")
table1_cvhist = summarize_cat(adsl, "CVHISTFL")

display(Markdown("### Table 1A. Age summary"))
display(styled_table(table1_age, "Age by treatment arm", precision=1))

display(Markdown("### Table 1B. Selected categorical baseline characteristics"))
baseline_cat = pd.concat([
    table1_sex.rename(columns={"SEX":"Category"}).assign(Characteristic="Sex"),
    table1_agegrp.rename(columns={"AGEGR1":"Category"}).assign(Characteristic="Age group"),
    table1_diab.rename(columns={"DIABFL":"Category"}).assign(Characteristic="Diabetes history flag"),
    table1_ckd.rename(columns={"CKDFL":"Category"}).assign(Characteristic="Chronic kidney disease flag"),
    table1_cvhist.rename(columns={"CVHISTFL":"Category"}).assign(Characteristic="Prior cardiovascular disease flag"),
], ignore_index=True)
baseline_cat = baseline_cat[["Characteristic","Category","TRT01A","n (%)"]]
display(styled_table(baseline_cat, "Categorical baseline characteristics", precision=1))

table1_age.to_csv(DATADIR / "tlf_table1_age.csv", index=False)
baseline_cat.to_csv(DATADIR / "tlf_table1_categorical.csv", index=False)

In [ ]:
# ================================================================
# 15. Table 2, blood pressure summary over time
# ================================================================

bp = advs.query("PARAMCD == 'SYSBP'").copy()
table2 = bp.groupby(["TRT01A","AVISITN","AVISIT"]).agg(
    N=("AVAL","count"),
    Mean_SBP=("AVAL","mean"),
    SD_SBP=("AVAL","std"),
    Mean_CHG=("CHG","mean"),
    SD_CHG=("CHG","std")
).reset_index().sort_values(["TRT01A","AVISITN"])

display(styled_table(table2, "Table 2. Systolic blood pressure by visit", precision=2))
table2.to_csv(DATADIR / "tlf_table2_sbp_by_visit.csv", index=False)

In [ ]:
# ================================================================
# 16. Table 3, adverse event summary
# ================================================================

n_by_arm = adsl.groupby("TRT01A")["USUBJID"].nunique().to_dict()

def subject_level_ae(condition, label):
    if len(adae) == 0:
        return pd.DataFrame()
    tmp = adae.query("TRTEMFL == 'Y'").copy()
    tmp = tmp[condition(tmp)]
    cnt = tmp.groupby("TRT01A")["USUBJID"].nunique().reset_index(name="n")
    cnt["N"] = cnt["TRT01A"].map(n_by_arm)
    cnt["Percent"] = 100 * cnt["n"] / cnt["N"]
    cnt["Category"] = label
    return cnt[["Category","TRT01A","n","N","Percent"]]

ae_summary = pd.concat([
    subject_level_ae(lambda x: x["AEDECOD"].notna(), "Any treatment-emergent adverse event"),
    subject_level_ae(lambda x: x["AESER"].eq("Y"), "Any serious adverse event"),
    subject_level_ae(lambda x: x["ASEV"].eq("SEVERE"), "Any severe adverse event"),
    subject_level_ae(lambda x: x["AREL"].isin(["POSSIBLY RELATED", "RELATED"]), "Any treatment-related adverse event"),
], ignore_index=True)

display(styled_table(ae_summary, "Table 3. Subject-level adverse event summary", precision=2))

ae_by_term = adae.query("TRTEMFL == 'Y'").groupby(["TRT01A","AEDECOD"]).agg(
    Subjects=("USUBJID","nunique"),
    Events=("AESEQ","count")
).reset_index()
ae_by_term["N"] = ae_by_term["TRT01A"].map(n_by_arm)
ae_by_term["Percent"] = 100 * ae_by_term["Subjects"] / ae_by_term["N"]
ae_by_term = ae_by_term.sort_values(["TRT01A","Subjects"], ascending=[True, False])
display(styled_table(ae_by_term.head(20), "Table 3B. Most common treatment-emergent adverse events", precision=2))

ae_summary.to_csv(DATADIR / "tlf_table3_ae_summary.csv", index=False)
ae_by_term.to_csv(DATADIR / "tlf_table3b_ae_by_term.csv", index=False)

In [ ]:
# ================================================================
# 17. Table 4, endpoint summary
# ================================================================

endpoint_summary = adtte.groupby("TRT01A").agg(
    N=("USUBJID","nunique"),
    Events=("CNSR", lambda x: (x == 0).sum()),
    Censored=("CNSR", lambda x: (x == 1).sum()),
    Mean_time_days=("AVAL","mean"),
    Median_time_days=("AVAL","median")
).reset_index()
endpoint_summary["Event_percent"] = 100 * endpoint_summary["Events"] / endpoint_summary["N"]

display(styled_table(endpoint_summary, "Table 4. Time-to-first cardiovascular endpoint summary", precision=2))
endpoint_summary.to_csv(DATADIR / "tlf_table4_endpoint_summary.csv", index=False)

In [ ]:
# ================================================================
# 18. Listings
# ================================================================

display(Markdown("### Listing 1. Treatment-emergent serious or severe adverse events"))
listing_ae = adae.query("TRTEMFL == 'Y' and (AESER == 'Y' or ASEV == 'SEVERE')").sort_values(["TRT01A","USUBJID","ASTDT"])
display(styled_table(listing_ae.head(30), "Serious or severe TEAE listing, first 30 records", precision=1))
listing_ae.to_csv(DATADIR / "listing_serious_or_severe_teae.csv", index=False)

display(Markdown("### Listing 2. Disposition"))
listing_ds = ds.merge(adsl[["USUBJID","TRT01A","AGE","SEX"]], on="USUBJID", how="left").sort_values(["TRT01A","USUBJID"])
display(styled_table(listing_ds.head(30), "Disposition listing, first 30 records", precision=1))
listing_ds.to_csv(DATADIR / "listing_disposition.csv", index=False)

In [ ]:
# ================================================================
# 19. Figures
# ================================================================

# Figure 1. Subject flow by disposition
flow = adsl.groupby(["TRT01A","EOSSTT"]).size().reset_index(name="Subjects")
flow_pivot = flow.pivot(index="EOSSTT", columns="TRT01A", values="Subjects").fillna(0)

plt.figure(figsize=(10, 6))
flow_pivot.plot(kind="bar", ax=plt.gca())
plt.title("Figure 1. Subject disposition by treatment arm")
plt.xlabel("Disposition status")
plt.ylabel("Number of subjects")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Treatment arm")
savefig("figure1_subject_disposition.png")

# Figure 2. Baseline SBP distribution
baseline_sbp = bp.query("AVISIT == 'BASELINE'")
plt.figure(figsize=(9, 6))
for arm, df in baseline_sbp.groupby("TRT01A"):
    plt.hist(df["AVAL"], bins=18, alpha=0.55, label=arm)
plt.title("Figure 2. Baseline systolic blood pressure distribution")
plt.xlabel("Systolic blood pressure, mmHg")
plt.ylabel("Number of subjects")
plt.legend(title="Treatment arm")
savefig("figure2_baseline_sbp_distribution.png")

# Figure 3. Mean SBP over visits
plt.figure(figsize=(10, 6))
for arm, df in table2.groupby("TRT01A"):
    plt.errorbar(df["AVISITN"], df["Mean_SBP"], yerr=df["SD_SBP"]/np.sqrt(df["N"]), marker="o", capsize=4, label=arm)
plt.title("Figure 3. Mean systolic blood pressure over time")
plt.xlabel("Analysis visit")
plt.ylabel("Mean systolic blood pressure, mmHg")
plt.xticks(table2["AVISITN"].unique(), table2.drop_duplicates("AVISITN").sort_values("AVISITN")["AVISIT"], rotation=30)
plt.legend(title="Treatment arm")
savefig("figure3_mean_sbp_over_time.png")

# Figure 4. Change from baseline at Month 12
m12 = bp.query("AVISIT == 'MONTH 12'").copy()
plt.figure(figsize=(10, 6))
data = [m12.loc[m12["TRT01A"] == arm, "CHG"].dropna() for arm in ARMS]
plt.boxplot(data, labels=ARMS)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Figure 4. Change from baseline in systolic blood pressure at Month 12")
plt.ylabel("Change from baseline, mmHg")
plt.xticks(rotation=20)
savefig("figure4_sbp_change_from_baseline_month12.png")

# Figure 5. Most common TEAEs
top_ae = adae.query("TRTEMFL == 'Y'").groupby("AEDECOD")["USUBJID"].nunique().sort_values(ascending=False).head(10).sort_values()
plt.figure(figsize=(9, 6))
top_ae.plot(kind="barh")
plt.title("Figure 5. Most common treatment-emergent adverse events")
plt.xlabel("Number of subjects")
plt.ylabel("Adverse event term")
savefig("figure5_common_teae.png")

# Figure 6. Lab abnormality rates
lab_abn = adlb.query("AVISIT != 'BASELINE'").assign(ABNFL=lambda x: np.where(x["ANRIND"].isin(["LOW","HIGH"]), 1, 0))
lab_abn_sum = lab_abn.groupby(["TRT01A","PARAM"]).agg(Records=("ABNFL","count"), Abnormal=("ABNFL","sum")).reset_index()
lab_abn_sum["Percent_abnormal"] = 100 * lab_abn_sum["Abnormal"] / lab_abn_sum["Records"]

plt.figure(figsize=(10, 6))
for i, arm in enumerate(ARMS):
    df = lab_abn_sum[lab_abn_sum["TRT01A"] == arm].sort_values("PARAM")
    x = np.arange(len(df)) + (i - 0.5) * 0.35
    plt.bar(x, df["Percent_abnormal"], width=0.35, label=arm)
plt.title("Figure 6. Post-baseline laboratory abnormality rates")
plt.xlabel("Laboratory parameter")
plt.ylabel("Percent of post-baseline records abnormal")
plt.xticks(np.arange(len(df)), df["PARAM"], rotation=25)
plt.legend(title="Treatment arm")
savefig("figure6_lab_abnormality_rates.png")

# Figure 7. Endpoint event percentage
plt.figure(figsize=(8, 6))
plt.bar(endpoint_summary["TRT01A"], endpoint_summary["Event_percent"])
plt.title("Figure 7. Cardiovascular endpoint event percentage")
plt.ylabel("Subjects with first MACE event, %")
plt.xticks(rotation=20)
savefig("figure7_endpoint_event_percentage.png")

## 7. Simple Inferential Summaries

This section adds simple analysis-style summaries. In a formal clinical trial, the statistical analysis plan would define the primary endpoint, estimand, populations, covariates, multiplicity strategy, handling of missing data, and statistical models.  

Here we calculate:

- Difference in Month 12 systolic blood pressure change.
- Crude endpoint risk ratio.
- Crude adverse-event relative frequencies.

These are for educational interpretation only.

In [ ]:
# ================================================================
# 20. Simple inferential-style summaries
# ================================================================

# Month 12 SBP change comparison
m12_summary = m12.groupby("TRT01A").agg(N=("CHG","count"), Mean_CHG=("CHG","mean"), SD_CHG=("CHG","std")).reset_index()
int_chg = m12_summary.loc[m12_summary["TRT01A"] == "INTENSIVE BP CONTROL", "Mean_CHG"].iloc[0]
std_chg = m12_summary.loc[m12_summary["TRT01A"] == "STANDARD BP CONTROL", "Mean_CHG"].iloc[0]
diff_chg = int_chg - std_chg

display(styled_table(m12_summary, "Month 12 SBP change summary", precision=2))
display(Markdown(f"**Estimated mean difference in Month 12 SBP change, intensive minus standard:** {diff_chg:.2f} mmHg."))

# Endpoint crude risk ratio
risk = endpoint_summary.set_index("TRT01A")
risk_int = risk.loc["INTENSIVE BP CONTROL", "Events"] / risk.loc["INTENSIVE BP CONTROL", "N"]
risk_std = risk.loc["STANDARD BP CONTROL", "Events"] / risk.loc["STANDARD BP CONTROL", "N"]
rr = risk_int / risk_std if risk_std > 0 else np.nan
display(Markdown(f"**Crude endpoint risk ratio, intensive versus standard:** {rr:.2f}. Values below 1.0 favour intensive control."))

# AE ratios by common event
ae_term_wide = ae_by_term.pivot(index="AEDECOD", columns="TRT01A", values="Percent").fillna(0)
if set(ARMS).issubset(ae_term_wide.columns):
    ae_term_wide["Percent_difference_intensive_minus_standard"] = ae_term_wide["INTENSIVE BP CONTROL"] - ae_term_wide["STANDARD BP CONTROL"]
display(styled_table(ae_term_wide.sort_values("Percent_difference_intensive_minus_standard", ascending=False).head(10), "AE percentage differences, intensive minus standard", precision=2))

## 8. Metadata and Define-XML-Like Documentation

A real regulatory package would include a machine-readable `define.xml`. This notebook creates a simplified metadata table showing dataset names, variable names, labels, origins, and derivations.

This demonstrates the Define-XML idea: reviewers need a map of what each dataset and variable means.

In [ ]:
# ================================================================
# 21. Simplified metadata table
# ================================================================

metadata_rows = [
    ("ADSL","USUBJID","Unique Subject Identifier","Character","SDTM DM","Copied from DM.USUBJID"),
    ("ADSL","TRT01A","Actual Treatment for Period 01","Character","SDTM DM/EX","Assigned from randomized treatment and exposure"),
    ("ADSL","TRTSDT","Treatment Start Date","Date","SDTM DM/EX","Derived from RFSTDTC/EXSTDTC"),
    ("ADSL","SAFFL","Safety Population Flag","Character","Derived","Y if subject received treatment"),
    ("ADVS","PARAMCD","Parameter Code","Character","SDTM VS","Mapped from VSTESTCD"),
    ("ADVS","AVAL","Analysis Value","Numeric","SDTM VS","Copied from VSSTRESN"),
    ("ADVS","BASE","Baseline Value","Numeric","Derived","Subject-level baseline value for same PARAMCD"),
    ("ADVS","CHG","Change from Baseline","Numeric","Derived","AVAL minus BASE"),
    ("ADLB","ANRIND","Analysis Normal Range Indicator","Character","SDTM LB","Copied from LBNRIND"),
    ("ADAE","TRTEMFL","Treatment-Emergent Analysis Flag","Character","Derived","Y if AE start date is on/after TRTSDT and within treatment risk window"),
    ("ADAE","AESER","Serious Event Flag","Character","SDTM AE","Copied from AE.AESER"),
    ("ADTTE","AVAL","Analysis Time","Numeric","Derived","Days from TRTSDT to event or censor date"),
    ("ADTTE","CNSR","Censor Indicator","Numeric","Derived","0 event, 1 censored"),
]
metadata = pd.DataFrame(metadata_rows, columns=["Dataset","Variable","Label","Type","Origin","Derivation"])
display(styled_table(metadata, "Simplified define.xml-like metadata table", precision=1))
metadata.to_csv(DATADIR / "define_like_metadata.csv", index=False)

## 9. Data Quality and Validation Checks

Clinical programming is not complete until datasets and outputs are validated. This section performs simple checks that imitate common programming review questions.

Examples:

- Does ADSL have one row per subject?
- Are all ADSL subjects present in DM?
- Are controlled terminology values valid?
- Are treatment-emergent adverse-event flags populated correctly?
- Are baseline values available for post-baseline analysis records?

In [ ]:
# ================================================================
# 22. Validation checks
# ================================================================

checks = []

checks.append(("ADSL one row per subject", adsl["USUBJID"].is_unique))
checks.append(("All ADSL subjects in SDTM DM", set(adsl["USUBJID"]).issubset(set(dm["USUBJID"]))))
checks.append(("All ADAE subjects in ADSL", set(adae["USUBJID"]).issubset(set(adsl["USUBJID"])) if len(adae) else True))
checks.append(("ADVS baseline value populated for all VS records", advs["BASE"].notna().all()))
checks.append(("ADLB baseline value populated for all LB records", adlb["BASE"].notna().all()))
checks.append(("ADAE TRTEMFL values valid", set(adae["TRTEMFL"].dropna().unique()).issubset({"Y","N"}) if len(adae) else True))
checks.append(("ADSL SAFFL values valid", set(adsl["SAFFL"].unique()).issubset({"Y","N"})))
checks.append(("ADSL ITTFL values valid", set(adsl["ITTFL"].unique()).issubset({"Y","N"})))
checks.append(("ADTTE censor values valid", set(adtte["CNSR"].unique()).issubset({0,1})))

validation = pd.DataFrame(checks, columns=["Check","Pass"])
validation["Result"] = np.where(validation["Pass"], "PASS", "FAIL")
display(styled_table(validation, "Validation report", precision=1))
validation.to_csv(DATADIR / "validation_report.csv", index=False)

if validation["Pass"].all():
    display(Markdown("✅ **All simplified validation checks passed.**"))
else:
    display(Markdown("⚠️ **Some validation checks failed. Review the validation table.**"))

## 10. Interpretation

The simulated dataset reproduces the broad educational pattern expected from a blood-pressure target trial:

1. The intensive arm achieves a larger reduction in systolic blood pressure by Month 12.
2. The endpoint event percentage is lower in the intensive arm in this simulated sample, consistent with a SPRINT-like direction of benefit.
3. Selected safety events such as hypotension, syncope, acute kidney injury, and electrolyte imbalance may be more frequent in the intensive arm, reflecting the clinical trade-off seen in intensive blood-pressure control studies.
4. SDTM-like domains provide organised tabulation data.
5. ADaM-like datasets provide analysis-ready variables, including baseline, change from baseline, treatment-emergent flags, safety flags, censoring variables, and traceability.
6. Tables, listings, and figures are generated from ADaM datasets, illustrating the clinical programming pathway from source-like data to review-ready outputs.

The most important lesson is not the exact simulated result. The most important lesson is the **data lifecycle**:

**Raw data → SDTM domains → ADaM analysis datasets → TLF/TLG outputs → interpretation and review.**

## 11. Files Created by This Notebook

When executed, this notebook saves outputs to:

`cdisc_adam_simulated_trial_outputs/`

Subfolders:

- `datasets/`, raw, SDTM-like, ADaM-like, TLF, listing, metadata, and validation CSV files.
- `figures/`, publication-style PNG figures.

These files can be used as a portfolio example for clinical data programming, ADS programming, statistical programming, and clinical data science interviews.